# Analysing Training Data Amount Effect

In [ ]:
%%capture
# Run to set environment variables if want to
# %env HF_TOKEN=

In [ ]:
%%capture
import os
# If getting 'Could not find project LASR_probe_gen' get key from https://wandb.ai/authorize and paste below
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_API_KEY"] = ""
import wandb
wandb_token = os.getenv("WANDB_API_KEY")
wandb.login(key=wandb_token)

### Increasing training data

In [ ]:
# Decide if want to increase all data or just positives or just negatives
increase_data = ["all", "positives", "negatives"][0]

# Balanced data, increasing positives and negatives
import probe_gen.probes as probes
from sklearn.metrics import classification_report
from probe_gen.config import ConfigDict
    
import torch
import numpy as np
from sklearn.utils import shuffle
import matplotlib.pyplot as plt

probe_type = ["mean", "attention_torch"][0]
behaviour = "sycophancy"
# datasource = "arguments"
datasource = "multichoice"
activations_model = "llama_3b"
response_model = "llama_3b"
# off_policy_model = "ministral_8b"
off_policy_model = "qwen_7b"
mode = "train"

# Load the best hyperparameters or set your own
dataset_name = None
cfg = ConfigDict.from_json(activations_model, probe_type, behaviour)

# Define subset sizes to test (must be even)
subset_sizes = list(range(100, 4000, 100))

# Fixed test generation method
test_generation_method = "on_policy"

# Load test dataset once (since it's fixed)
used_model_test = response_model if test_generation_method != "off_policy" else off_policy_model
activations_tensor_test, attention_mask_test, labels_tensor_test = probes.load_hf_activations_at_layer(
    behaviour, datasource, activations_model, used_model_test, test_generation_method, "test", cfg.layer, and_labels=True)
if "mean" in probe_type:
    activations_tensor_test = probes.MeanAggregation()(activations_tensor_test, attention_mask_test)
_, _, test_dataset = probes.create_activation_datasets(
    activations_tensor_test, labels_tensor_test, splits=[0, 0, 1000])

# Store results for each generation method
all_results = {}

# Iterate over different training generation methods
for generation_method in ["on_policy", "incentivised", "prompted", "off_policy"]:
    print(f"\n{'='*60}")
    print(f"Training with generation method: {generation_method}")
    print(f"{'='*60}\n")
    
    # Load training activations for this generation method
    used_model = response_model if generation_method != "off_policy" else off_policy_model
    activations_tensor, attention_mask, labels_tensor = probes.load_hf_activations_at_layer(
        behaviour, datasource, activations_model, used_model, generation_method, mode, cfg.layer, and_labels=True)
    if "mean" in probe_type:
        activations_tensor = probes.MeanAggregation()(activations_tensor, attention_mask)
    
    # Split indices by class
    pos_indices = (labels_tensor == 1).nonzero(as_tuple=True)[0]
    neg_indices = (labels_tensor == 0).nonzero(as_tuple=True)[0]
    
    results = {}
    
    for n in subset_sizes:
        half = n // 2
        # Randomly sample equal numbers from each class
        if increase_data == "all":
            pos_cut = half
            neg_cut = half
        elif increase_data == "positives":
            pos_cut = half
            neg_cut = 4000
        elif increase_data == "negatives":
            pos_cut = 4000
            neg_cut = half
        pos_sample = pos_indices[torch.randperm(len(pos_indices))[:pos_cut]]
        neg_sample = neg_indices[torch.randperm(len(neg_indices))[:neg_cut]]
        subset_idx = torch.cat([pos_sample, neg_sample])
        subset_idx = subset_idx[torch.randperm(len(subset_idx))]  # shuffle order
    
        # Select activations and labels
        activ_subset = activations_tensor[subset_idx]
        label_subset = labels_tensor[subset_idx]
    
        # Create datasets (e.g., 80% train, 20% val)
        n_train = int(0.8 * len(label_subset))
        n_val = len(label_subset) - n_train
        train_dataset, val_dataset, _ = probes.create_activation_datasets(
            activ_subset, label_subset, splits=[n_train, n_val, 0]
        )
    
        # Initialise and train probe
        if probe_type == "mean":
            probe = probes.SklearnLogisticProbe(cfg)
        elif probe_type == "mean_torch":
            probe = probes.TorchLinearProbe(cfg)
        elif probe_type == "attention_torch":
            probe = probes.TorchAttentionProbe(cfg)
    
        probe.fit(train_dataset, val_dataset)
    
        # Evaluate on the fixed test set
        eval_dict, _, _ = probe.eval(test_dataset)
        results[n] = eval_dict["roc_auc"]
    
        print(f"Subset size {n} → ROC-AUC: {eval_dict['roc_auc']:.3f}")
    
    # Store results for this generation method
    all_results[generation_method] = results

# Plot all results on the same graph
plt.figure(figsize=(10, 6))
for gen_method, results in all_results.items():
    plt.plot(list(results.keys()), list(results.values()), marker='o', label=gen_method)

plt.xlabel("Training set size")
plt.ylabel("Test ROC-AUC")
plt.title(f"Data efficiency for {probe_type} probe on {behaviour}\n(Test: {test_generation_method})")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.ylim(0.5, 1.0)
plt.show()

arguments/llama_3b/llama_3b_on_policy_te(…):   0%|          | 0.00/1.22G [00:00<?, ?B/s]

llama_3b_on_policy_test.jsonl: 0.00B [00:00, ?B/s]


Training with generation method: on_policy



arguments/llama_3b/llama_3b_on_policy_tr(…):   0%|          | 0.00/4.94G [00:00<?, ?B/s]

llama_3b_on_policy_train.jsonl: 0.00B [00:00, ?B/s]

...

Subset size 100 → ROC-AUC: 0.589
Subset size 200 → ROC-AUC: 0.602
Subset size 300 → ROC-AUC: 0.629
Subset size 400 → ROC-AUC: 0.664
Subset size 500 → ROC-AUC: 0.677
Subset size 600 → ROC-AUC: 0.646
Subset size 700 → ROC-AUC: 0.705
Subset size 800 → ROC-AUC: 0.701
Subset size 900 → ROC-AUC: 0.684
Subset size 1000 → ROC-AUC: 0.651
Subset size 1100 → ROC-AUC: 0.683
Subset size 1200 → ROC-AUC: 0.730
Subset size 1300 → ROC-AUC: 0.691
Subset size 1400 → ROC-AUC: 0.717
Subset size 1500 → ROC-AUC: 0.677
Subset size 1600 → ROC-AUC: 0.708
Subset size 1700 → ROC-AUC: 0.703
Subset size 1800 → ROC-AUC: 0.701
Subset size 1900 → ROC-AUC: 0.712
Subset size 2000 → ROC-AUC: 0.717
Subset size 2100 → ROC-AUC: 0.714
Subset size 2200 → ROC-AUC: 0.689
Subset size 2300 → ROC-AUC: 0.698
Subset size 2400 → ROC-AUC: 0.678
Subset size 2500 → ROC-AUC: 0.696
Subset size 2600 → ROC-AUC: 0.734
Subset size 2700 → ROC-AUC: 0.718
Subset size 2800 → ROC-AUC: 0.701
Subset size 2900 → ROC-AUC: 0.697
Subset size 3000 → ROC-

arguments/llama_3b/llama_3b_prompted_tra(…):   0%|          | 0.00/4.53G [00:00<?, ?B/s]

llama_3b_prompted_train.jsonl: 0.00B [00:00, ?B/s]

Subset size 100 → ROC-AUC: 0.476
Subset size 200 → ROC-AUC: 0.513
Subset size 300 → ROC-AUC: 0.622
Subset size 400 → ROC-AUC: 0.577
Subset size 500 → ROC-AUC: 0.494
Subset size 600 → ROC-AUC: 0.591
Subset size 700 → ROC-AUC: 0.608
Subset size 800 → ROC-AUC: 0.608
Subset size 900 → ROC-AUC: 0.608
Subset size 1000 → ROC-AUC: 0.555
Subset size 1100 → ROC-AUC: 0.582
Subset size 1200 → ROC-AUC: 0.591
Subset size 1300 → ROC-AUC: 0.554
Subset size 1400 → ROC-AUC: 0.622
Subset size 1500 → ROC-AUC: 0.604
Subset size 1600 → ROC-AUC: 0.609
Subset size 1700 → ROC-AUC: 0.572
Subset size 1800 → ROC-AUC: 0.621
Subset size 1900 → ROC-AUC: 0.597
Subset size 2000 → ROC-AUC: 0.606
Subset size 2100 → ROC-AUC: 0.599
Subset size 2200 → ROC-AUC: 0.605
Subset size 2300 → ROC-AUC: 0.598
Subset size 2400 → ROC-AUC: 0.618
Subset size 2500 → ROC-AUC: 0.597
Subset size 2600 → ROC-AUC: 0.614
Subset size 2700 → ROC-AUC: 0.619
Subset size 2800 → ROC-AUC: 0.625
Subset size 2900 → ROC-AUC: 0.597
Subset size 3000 → ROC-

EntryNotFoundError: 404 Client Error. (Request ID: Root=1-69405897-10025691078e9cee5f035125;ffd15140-2efb-48b5-ae32-f1104d653c89)

Entry Not Found for url: https://huggingface.co/datasets/lasrprobegen/sycophancy-activations/resolve/main/arguments/llama_3b/ministral_8b_on_policy_train_layer_12.pkl.